In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [2]:
import os

In [3]:
## OpenAI
os.environ["OPENAI_API_KEY"]=""


In [4]:
# Load dos modelos (Embeddings e LLM)

embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name ="gpt-3.5-turbo", max_tokens = 200)


In [5]:
# Carregar PDF

pdf_link = "lei_nacionalidade_pt.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [6]:
# Separar em Chunks (pedaços de documento)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 20,
    length_function = len,
    add_start_index = True,
)

chunks = text_splitter.split_documents(pages)

In [7]:
# Salvar no Vector DB - Chroma

db = Chroma.from_documents(chunks, embedding=embeddings_model, persist_directory="text_index")

In [8]:
# Carregar DB
vectordb = Chroma(persist_directory="text_index", embedding_function=embeddings_model)

# Load Retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Prompt
system_prompt = (
    "Você é um assistente especializado na lei de nacionalidade de Portugal. "
    "Use o contexto fornecido para responder à pergunta. "
    "Se não souber a resposta, diga que não sabe.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Construção da cadeia de prompt para chamada do LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [13]:
def ask(question):
    response = rag_chain.invoke({"input": question})
    return response["answer"]

In [14]:
user_question = input("User: ")
answer = ask(user_question)
print("Answer: ", answer)

User:  Quem são considerados portugueses de origem segundo a lei de nacionalidade?


Answer:  Os portugueses de origem, de acordo com a lei de nacionalidade de Portugal, são considerados aqueles que se enquadram em uma das seguintes condições:

a) Filhos de mãe ou pai português, nascidos no território português;

b) Filhos de mãe ou pai português, nascidos no estrangeiro, desde que o progenitor português aí se encontre ao serviço do Estado português;

c) Filhos de mãe ou pai português, desde que os progenitores não tenham perdido a nacionalidade portuguesa;

d) Filhos de mãe ou pai português, nascidos no estrangeiro, desde que declarem que querem ser portugueses.
